In [1]:
import numpy as np
import xcdat as xc
import scipy
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt 
from cdo import *   # python version
import scipy.stats as stats
import os
import glob
import cmasher as cmr
import xarray as xr

sys.path.append('../../functions/')
from lag_linregress import *
from monthly_departures import *
from MCA import *

In [2]:
# Reload edited Python modules automatically before running subsequent cells.
%load_ext autoreload
%autoreload 2

import sys

# Add the project folder only once, even if this cell is rerun.
functions_path = "/scratch/leiff/MCA/amip_clean/project_functions/"
if functions_path not in sys.path:
    sys.path.append(functions_path)

from MCA_cov import *
from significance import *
from plotting import *
from diagnostics import *
from taylor_setup import *

In [3]:
# %% 10. Load-only cell: usable in a fresh kernel without rerunning the calculations
# Only load pickle-containing .npy dictionaries that you created/trust.
import numpy as np
from pathlib import Path

overlap_save_dir        = Path("/scratch/leiff/MCA/amip_clean/data/maps_2000_2014_covImp//overlap_core_200003_201412")
overlap_member_values   = np.load(overlap_save_dir / "overlap_member_values.npy", allow_pickle=True).item()
overlap_model_means     = np.load(overlap_save_dir / "overlap_model_means.npy", allow_pickle=True).item()
overlap_MMM             = np.load(overlap_save_dir / "overlap_MMM.npy", allow_pickle=True).item()
overlap_observations    = np.load(overlap_save_dir / "overlap_observations.npy", allow_pickle=True).item()
overlap_model_n_valid   = np.load(overlap_save_dir / "overlap_model_n_valid.npy", allow_pickle=True).item()
overlap_MMM_n_valid     = np.load(overlap_save_dir / "overlap_MMM_n_valid.npy", allow_pickle=True).item()
overlap_model_names     = np.load(overlap_save_dir / "overlap_model_names.npy", allow_pickle=True).item()
overlap_member_ids      = np.load(overlap_save_dir / "overlap_member_ids.npy", allow_pickle=True).item()
overlap_coordinates     = np.load(overlap_save_dir / "overlap_coordinates.npy", allow_pickle=True).item()
overlap_quantity_info   = np.load(overlap_save_dir / "overlap_quantity_info.npy", allow_pickle=True).item()
overlap_sources         = np.load(overlap_save_dir / "overlap_sources.npy", allow_pickle=True).item()
overlap_metadata        = np.load(overlap_save_dir / "overlap_metadata.npy", allow_pickle=True).item()

In [4]:
# List out quantities in the data structure
for key, list_obj in overlap_quantity_info.items():
    print(f"--- {key} ---")
    # print(*list_obj, sep='\n')
    print(list_obj)
    print()  # Adds an empty line between groups


--- corr_Ti_Ri ---
{'kind': 'map', 'units': '1', 'meaning': 'Local T versus local R correlation'}

--- corr_T_Ri ---
{'kind': 'map', 'units': '1', 'meaning': 'Global T versus local R correlation'}

--- corr_Ti_R ---
{'kind': 'map', 'units': '1', 'meaning': 'Local T versus global R correlation'}

--- corr_T_R ---
{'kind': 'scalar', 'units': '1', 'meaning': 'Global T versus global R correlation'}

--- sigma_Ti ---
{'kind': 'map', 'units': 'T input units', 'meaning': 'Local temporal T SD'}

--- sigma_Ri ---
{'kind': 'map', 'units': 'R input units', 'meaning': 'Local temporal R SD'}

--- sigma_T ---
{'kind': 'scalar', 'units': 'T input units', 'meaning': 'Global temporal T SD'}

--- sigma_R ---
{'kind': 'scalar', 'units': 'R input units', 'meaning': 'Global temporal R SD'}

--- T_detrended ---
{'kind': 'series', 'units': 'T input units', 'meaning': 'Global detrended T anomaly'}

--- R_detrended ---
{'kind': 'series', 'units': 'R input units', 'meaning': 'Global detrended R anomaly'}

--- T

In [5]:
# %% 1. Scoring helpers: reuse compare_maps for both space and time
# First load your overlap_* archive and run/import your existing compare_maps.
# No climate files are reread, and no detrending or standardization is repeated.
import numpy as np
from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
import inspect

if not callable(globals().get("compare_maps")):
    raise NameError("Run/import compare_maps from your existing comparison-function cell first.")

def compare_timeseries(x, y, weights=None, *, mask=None):
    """Exactly the compare_maps scores, reducing a pair of aligned 1-D time series.

    x is observation; y is model. None weights means equally weighted MONTHS.
    The retained key pattern_correlation means temporal correlation here;
    contrast_ratio is the temporal SD ratio. No lags or significance tests are used.
    Moments use normalized weights (ddof=0), just as in the map comparison.
    """
    if np.ndim(x) != 1 or np.ndim(y) != 1:
        raise ValueError("compare_timeseries requires two one-dimensional, already-aligned series.")
    return compare_maps(x, y, weights=weights, mask=mask)

def compare_overlap_scalars(x, y):
    """Meaningful comparisons of one observation scalar with one model scalar.

    RMSE for one pair is simply absolute error. value_ratio retains the sign y/x;
    normalized errors divide by abs(x). No pattern correlation, SD, or CCC is
    assigned to a single pair. In particular, corr_T_R is itself a scalar diagnostic.
    """
    if np.ndim(x) != 0 or np.ndim(y) != 0:
        raise ValueError("Expected one scalar observation and one scalar model value.")
    x, y = float(x), float(y)
    result = {key: np.nan for key in ["value_x", "value_y", "bias", "rmse", "value_ratio",
                                     "rmse_over_rms_x", "bias_over_rms_x"]}
    result.update(n_valid=0, valid_weight_fraction=0.)
    if np.isfinite(x) and np.isfinite(y):
        difference = y - x
        result.update(value_x=x, value_y=y, bias=difference, rmse=abs(difference),
                      value_ratio=y / x if x != 0 else np.nan,
                      rmse_over_rms_x=abs(difference) / abs(x) if x != 0 else np.nan,
                      bias_over_rms_x=difference / abs(x) if x != 0 else np.nan,
                      n_valid=1, valid_weight_fraction=1.)
    return result

def score_overlap_pair(x, y, kind, weights=None, mask=None):
    """Select map/series/scalar scoring; preserve entirely undefined pairs as NaNs."""
    if kind == "scalar":
        return compare_overlap_scalars(x, y)
    if kind not in ("map", "series"):
        raise ValueError(f"Unknown quantity kind: {kind}")
    x = np.ma.asarray(x, dtype=float).filled(np.nan)
    y = np.ma.asarray(y, dtype=float).filled(np.nan)
    if x.ndim == 0 or x.shape != y.shape or (kind == "series" and x.ndim != 1):
        raise ValueError("Expected matching non-scalar shapes; series must be one-dimensional.")

    # Some correlation maps can be wholly undefined (e.g. a constant global index).
    # Keep those members in the archive with missing scores and zero coverage.
    # Invalid shapes/negative weights/empty requested domains still raise errors.
    w = np.ma.asarray(1. if weights is None else weights, dtype=float).filled(np.nan)
    w = np.broadcast_to(w, x.shape)
    include = np.ma.asarray(True if mask is None else mask, dtype=bool).filled(False)
    include = np.broadcast_to(include, x.shape)
    if np.any(np.isfinite(w) & (w < 0)):
        raise ValueError("Comparison weights cannot be negative.")
    support = include & np.isfinite(w) & (w > 0)
    if not support.any():
        raise ValueError("The requested comparison domain has no positive finite weights.")
    if not np.any(support & np.isfinite(x) & np.isfinite(y)):
        # Read the score names from the actual function, rather than maintaining a
        # second list that could miss a newly added compare_maps score.
        probe = np.array([0., 1., 2.])
        result = {key: np.nan for key in compare_maps(probe, probe)}
        result.update(n_valid=0, valid_weight_fraction=0.)
        return result
    function = compare_timeseries if kind == "series" else compare_maps
    return function(x, y, weights=w, mask=include)

def average_overlap_scores(score_arrays):
    """Average each score over finite contributors, also recording their counts."""
    averages, counts = {}, {}
    for key, values in score_arrays.items():
        values = np.asarray(values, dtype=float)
        finite = np.isfinite(values)
        averages[key] = float(values[finite].mean()) if finite.any() else np.nan
        counts[key] = int(finite.sum())
    return averages, counts




In [6]:
# %% 2. Select quantities and comparison weights; initialize the new structures
# All registered quantities are included automatically: the default archive has
# five maps, four series, and three scalars. Registered extensions are included too.
# This does not create additional covariance variants from correlation/SD products;
# reconstruct/register those member maps and their means first if you want to score them.
overlap_score_quantities = list(overlap_quantity_info)
overlap_score_model_names = deepcopy(overlap_model_names)
# print(overlap_score_model_names)
overlap_score_member_ids = deepcopy(overlap_member_ids)
# print(overlap_score_member_ids)

# Map scores use area weights within the archive's local-map domain. To restrict
# scoring further, intersect this mask with a fixed regional mask. Do NOT use each
# model's significance mask; that would make their scored domains differ by design.
# Masks/weights are NumPy arrays in the SAVED coordinate order; no regridding occurs.
overlap_score_map_weights = np.asarray(overlap_coordinates["base_weights"], dtype=float).copy()
overlap_score_map_mask = np.asarray(overlap_coordinates["local_support"], dtype=bool).copy()
# overlap_score_map_mask &= your_region_mask

# Time is treated as the comparison dimension, with one equal weight per month.
# A different weighting (e.g. month duration) is an explicit choice, not the default.
# Calendar alignment was performed when creating the archive; retain that time order.
overlap_score_time_weights = np.ones(len(overlap_coordinates["time"]), dtype=float)
overlap_score_time_mask = np.ones(len(overlap_coordinates["time"]), dtype=bool)
# overlap_score_time_mask &= overlap_coordinates["time"] >= np.datetime64("2005-01")

# Keep performance OF a mean separate from the mean OF member performances.
# The first three containers use [experiment][quantity][model][score].
# The two MMM containers use [experiment][quantity][score], without a model key.
overlap_member_scores, overlap_mean_member_scores, overlap_model_mean_scores = {}, {}, {}
overlap_MMM_scores, overlap_MMM_mean_member_scores = {}, {}
overlap_mean_member_score_n_valid, overlap_MMM_mean_member_score_n_valid = {}, {}

if all(exp in overlap_score_model_names for exp in ["amip_hist", "historical", "historical_matched"]):
    shared = set(overlap_score_model_names["amip_hist"]) & set(overlap_score_model_names["historical"])
    if not shared or set(overlap_score_model_names["historical_matched"]) != shared:
        raise ValueError("historical_matched must contain exactly the nonempty AMIP–historical intersection.")




In [7]:
# %% 3. Score every member, every mean field, and each MMM against observations
for experiment, models in overlap_score_model_names.items():
    if not models or len(models) != len(set(models)):
        raise ValueError(f"{experiment}: model names must be nonempty and unique.")
    for output in [overlap_member_scores, overlap_mean_member_scores, overlap_model_mean_scores,
                   overlap_MMM_scores, overlap_MMM_mean_member_scores,
                   overlap_mean_member_score_n_valid, overlap_MMM_mean_member_score_n_valid]:
        output[experiment] = {}

    for quantity in overlap_score_quantities:
        kind = overlap_quantity_info[quantity]["kind"]
        observation = np.ma.asarray(overlap_observations[quantity], dtype=float).filled(np.nan)
        means = np.ma.asarray(overlap_model_means[experiment][quantity], dtype=float).filled(np.nan)
        if means.shape != (len(models),) + observation.shape:
            raise ValueError(f"{experiment}/{quantity}: model-mean shape disagrees with names/observations.")
        if kind == "map":
            weights, mask = overlap_score_map_weights, overlap_score_map_mask
        elif kind == "series":
            weights, mask = overlap_score_time_weights, overlap_score_time_mask
            if observation.shape != overlap_score_time_weights.shape:
                raise ValueError(f"{quantity}: series length disagrees with the saved time coordinate.")
        elif kind == "scalar":
            weights, mask = None, None
        else:
            raise ValueError(f"{quantity}: unknown registered kind {kind}.")
        for output in [overlap_member_scores, overlap_mean_member_scores, overlap_model_mean_scores,
                       overlap_mean_member_score_n_valid]:
            output[experiment][quantity] = {}
        print(f"{experiment} / {quantity} ({kind}): {len(models)} models")

        for m, model in enumerate(models):
            members = np.ma.asarray(overlap_member_values[experiment][quantity][model], dtype=float).filled(np.nan)
            member_ids = overlap_score_member_ids[experiment][model]
            if len(member_ids) == 0 or members.shape != (len(member_ids),) + observation.shape:
                raise ValueError(f"{experiment}/{quantity}/{model}: member shape disagrees with IDs/observations.")

            # Observations are ALWAYS x; each individual model member is y. Map
            # scores reduce space; series scores reduce time; scalars are single pairs.
            # Keep the full returned score dictionary, including CCC and coverage.
            results = []
            for member_value in members:
                results.append(score_overlap_pair(observation, member_value, kind, weights, mask))
            arrays = {key: np.asarray([result[key] for result in results]) for key in results[0]}
            overlap_member_scores[experiment][quantity][model] = arrays

            # Mean member performance and the number of finite member scores.
            # Correlations are averaged arithmetically (no Fisher-z transform).
            averages, counts = average_overlap_scores(arrays)
            overlap_mean_member_scores[experiment][quantity][model] = averages
            overlap_mean_member_score_n_valid[experiment][quantity][model] = counts

            # Separately score the saved model-mean map/series/scalar. For example,
            # RMSE(mean member series, obs) is NOT mean(RMSE(member series, obs)).
            overlap_model_mean_scores[experiment][quantity][model] = score_overlap_pair(
                observation, means[m], kind, weights, mask)

        # Score the experiment's saved MMM field/scalar, using its own model group.
        overlap_MMM_scores[experiment][quantity] = score_overlap_pair(
            observation, overlap_MMM[experiment][quantity], kind, weights, mask)

        # Also store the MMM OF MEMBER SCORES: average within each model first, then
        # average across models with a defined mean score. Large ensembles get no
        # extra weight. Counts tell you how many models support each average.
        model_score_arrays = {key: np.asarray([overlap_mean_member_scores[experiment][quantity][model][key]
                                              for model in models]) for key in arrays}
        averages, counts = average_overlap_scores(model_score_arrays)
        overlap_MMM_mean_member_scores[experiment][quantity] = averages
        overlap_MMM_mean_member_score_n_valid[experiment][quantity] = counts

# Preserve the conventions needed to interpret the scores after loading them later.
# The original fields, member IDs, and coordinates must remain aligned; scoring does
# not attempt to infer alignment from equal shapes or reconcile different units.
try:
    comparison_source = inspect.getsource(compare_maps)
except (OSError, TypeError):
    comparison_source = None
first_experiment = next(iter(overlap_score_model_names))
overlap_score_metadata = {
    "schema_version": 1, "created_utc": datetime.now(timezone.utc).isoformat(),
    "quantity_info": {q: deepcopy(overlap_quantity_info[q]) for q in overlap_score_quantities},
    "score_keys": {q: list(overlap_MMM_scores[first_experiment][q]) for q in overlap_score_quantities},
    "coordinates": {key: np.asarray(overlap_coordinates[key]).copy() for key in ["time", "lat", "lon"]},
    "map_weights": overlap_score_map_weights.copy(), "map_mask": overlap_score_map_mask.copy(),
    "time_weights": overlap_score_time_weights.copy(), "time_mask": overlap_score_time_mask.copy(),
    "source_archive_metadata": deepcopy(overlap_metadata), "compare_maps_source": comparison_source,
    "reference": "x = observations; y = model; bias = y - x; amplitude ratios = model / observation",
    "score_moments": "Normalized weighted moments, ddof=0, for both space and time",
    "missing_values": "Pairwise finite support per comparison; entirely undefined pairs give NaN scores and zero coverage",
    "aggregation": "Arithmetic finite member-score means, then equal-model means; no Fisher-z transform",
    "coverage": "n_valid counts paired cells/months (or 0/1 for scalars); valid_weight_fraction is retained domain weight",
    "summary_counts": "Separate *_score_n_valid containers count finite member/model SCORES, not cells/months",
    "series_interpretation": "pattern_correlation is temporal correlation; contrast_ratio is temporal SD ratio",
    "scalar_interpretation": "RMSE is absolute error; value_ratio = y/x; no pattern correlation, contrast, or CCC assigned",
}
print("Finished scoring all registered quantities; the original archives were not modified.")




amip_hist / corr_Ti_Ri (map): 16 models


amip_hist / corr_T_Ri (map): 16 models
amip_hist / corr_Ti_R (map): 16 models
amip_hist / corr_T_R (scalar): 16 models
amip_hist / sigma_Ti (map): 16 models
amip_hist / sigma_Ri (map): 16 models
amip_hist / sigma_T (scalar): 16 models
amip_hist / sigma_R (scalar): 16 models
amip_hist / T_detrended (series): 16 models
amip_hist / R_detrended (series): 16 models
amip_hist / T_trend_in (series): 16 models
amip_hist / R_trend_in (series): 16 models
amip_hist / cov_Zi_R (map): 16 models
amip_hist / cov_Ti_R (map): 16 models
amip_hist / cov_T_Qi (map): 16 models
amip_hist / cov_T_Ri (map): 16 models
amip_hist / cov_Zi_Ri (map): 16 models
amip_hist / cov_Ti_Qi (map): 16 models
amip_hist / cov_Ti_Ri (map): 16 models
historical / corr_Ti_Ri (map): 53 models
historical / corr_T_Ri (map): 53 models
historical / corr_Ti_R (map): 53 models
historical / corr_T_R (scalar): 53 models
historical / sigma_Ti (map): 53 models
historical / sigma_Ri (map): 53 models
historical / sigma_T (scalar): 53 models


In [8]:
overlap_score_metadata

{'schema_version': 1,
 'created_utc': '2026-09-12T18:17:50.442709+00:00',
 'quantity_info': {'corr_Ti_Ri': {'kind': 'map',
   'units': '1',
   'meaning': 'Local T versus local R correlation'},
  'corr_T_Ri': {'kind': 'map',
   'units': '1',
   'meaning': 'Global T versus local R correlation'},
  'corr_Ti_R': {'kind': 'map',
   'units': '1',
   'meaning': 'Local T versus global R correlation'},
  'corr_T_R': {'kind': 'scalar',
   'units': '1',
   'meaning': 'Global T versus global R correlation'},
  'sigma_Ti': {'kind': 'map',
   'units': 'T input units',
   'meaning': 'Local temporal T SD'},
  'sigma_Ri': {'kind': 'map',
   'units': 'R input units',
   'meaning': 'Local temporal R SD'},
  'sigma_T': {'kind': 'scalar',
   'units': 'T input units',
   'meaning': 'Global temporal T SD'},
  'sigma_R': {'kind': 'scalar',
   'units': 'R input units',
   'meaning': 'Global temporal R SD'},
  'T_detrended': {'kind': 'series',
   'units': 'T input units',
   'meaning': 'Global detrended T anoma

In [9]:
# %% 4. Quick indexing examples
experiment, quantity = "historical_matched", "R_detrended"
model = overlap_score_model_names[experiment][0]

# Individual-member performance and mean member performance for ONE model.
print(model, "member RMSE:", overlap_member_scores[experiment][quantity][model]["rmse"])
print("Mean member RMSE:", overlap_mean_member_scores[experiment][quantity][model]["rmse"])

# Keep these separate: the average member's performance versus performance of the
# smoother MMM series. The same indexing applies to a map such as corr_Ti_R.
print("Equal-model mean of member RMSE:", overlap_MMM_mean_member_scores[experiment][quantity]["rmse"])
print("RMSE of MMM series:", overlap_MMM_scores[experiment][quantity]["rmse"])




BCC-CSM2-MR member RMSE: [0.83094094 0.97679647 0.8610245 ]
Mean member RMSE: 0.8895873013948205
Equal-model mean of member RMSE: 0.9070137914354279
RMSE of MMM series: 0.6535104006768778


In [10]:
# %% 5. Save the NEW score dictionaries, separate from your previous five-map scores
overlap_score_save_dir = Path("/scratch/leiff/MCA/amip_clean/data/maps_2000_2014_covImp/overlap_core_200003_201412/vs_obs")
overlap_score_overwrite = False  # Change deliberately, or choose a new directory for a different domain.
overlap_score_save_items = {
    "overlap_member_scores": overlap_member_scores,
    "overlap_mean_member_scores": overlap_mean_member_scores,
    "overlap_model_mean_scores": overlap_model_mean_scores,
    "overlap_MMM_scores": overlap_MMM_scores,
    "overlap_MMM_mean_member_scores": overlap_MMM_mean_member_scores,
    "overlap_mean_member_score_n_valid": overlap_mean_member_score_n_valid,
    "overlap_MMM_mean_member_score_n_valid": overlap_MMM_mean_member_score_n_valid,
    "overlap_score_model_names": overlap_score_model_names,
    "overlap_score_member_ids": overlap_score_member_ids,
    "overlap_score_metadata": overlap_score_metadata,
}
existing = [overlap_score_save_dir / f"{name}.npy" for name in overlap_score_save_items
            if (overlap_score_save_dir / f"{name}.npy").exists()]
if existing and not overlap_score_overwrite:
    raise FileExistsError(f"Score files already exist; choose another directory or enable overwrite: {existing}")
overlap_score_save_dir.mkdir(parents=True, exist_ok=True)
for name, values in overlap_score_save_items.items():
    np.save(overlap_score_save_dir / f"{name}.npy", values, allow_pickle=True)
print(f"Saved {len(overlap_score_save_items)} dictionaries to {overlap_score_save_dir}")




Saved 10 dictionaries to /scratch/leiff/MCA/amip_clean/data/maps_2000_2014_covImp/overlap_core_200003_201412/vs_obs


In [ ]:
# %% 6. Load-only cell: works in a fresh kernel, without raw fields or scoring functions
# These .npy dictionaries contain pickle data; load only files you created/trust.
import numpy as np
from pathlib import Path
stop
overlap_score_save_dir = Path("/scratch/leiff/MCA/amip_clean/data/maps_2000_2014_covImp/overlap_core_200003_201412/vs_obs")
overlap_member_scores = np.load(overlap_score_save_dir / "overlap_member_scores.npy", allow_pickle=True).item()
overlap_mean_member_scores = np.load(overlap_score_save_dir / "overlap_mean_member_scores.npy", allow_pickle=True).item()
overlap_model_mean_scores = np.load(overlap_score_save_dir / "overlap_model_mean_scores.npy", allow_pickle=True).item()
overlap_MMM_scores = np.load(overlap_score_save_dir / "overlap_MMM_scores.npy", allow_pickle=True).item()
overlap_MMM_mean_member_scores = np.load(overlap_score_save_dir / "overlap_MMM_mean_member_scores.npy", allow_pickle=True).item()
overlap_mean_member_score_n_valid = np.load(overlap_score_save_dir / "overlap_mean_member_score_n_valid.npy", allow_pickle=True).item()
overlap_MMM_mean_member_score_n_valid = np.load(overlap_score_save_dir / "overlap_MMM_mean_member_score_n_valid.npy", allow_pickle=True).item()
overlap_score_model_names = np.load(overlap_score_save_dir / "overlap_score_model_names.npy", allow_pickle=True).item()
overlap_score_member_ids = np.load(overlap_score_save_dir / "overlap_score_member_ids.npy", allow_pickle=True).item()
overlap_score_metadata = np.load(overlap_score_save_dir / "overlap_score_metadata.npy", allow_pickle=True).item()
